# Member 3: WIS, patient-cluster bootstrap and limitations

Run Members 1 and 2 first. Sensitivity uses validation data. The predeclared final policy is evaluated on test data without choosing settings to improve test results.

In [ ]:
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, display

root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'main_pipeline.py').is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project root or a member folder")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from main_pipeline import load_configs, prepare_data, train_model, evaluate_model

output = Path(os.environ.get('ASS2_OUTPUT_DIR', root)).resolve()
print("Source: synthetic coursework demo; no clinical efficacy claim.")
print("Output:", output)


## 1. Evaluate the saved policy

WIS normalizes trajectory log-weights without flooring small values. The target is the softened policy, not the deterministic training policy. Behavior probabilities are estimates; smoothing does not establish causal identification or true overlap.

In [ ]:
results = evaluate_model(output)
metrics = json.loads((output / 'artifacts/evaluation_metrics.json').read_text())
display(metrics)
if results['low_ess']:
    print("WARNING: Low ESS. Bootstrap confidence intervals do not establish policy superiority.")


## 2. Uncertainty and the estimand

The baseline is the observed mean discounted episode return. The target uses WIS over episodes. Bootstrap samples patients and retains every ICU stay per sampled patient. An all-zero-support replicate is counted as invalid, not replaced by zero.

In [ ]:
pd.DataFrame({
    'observed behavior': [results['point_clinician'], results['clin_median'], results['clin_ci_lower'], results['clin_ci_upper']],
    'soft target WIS': [results['point_wis'], results['wis_median'], results['wis_ci_lower'], results['wis_ci_upper']],
}, index=['point', 'bootstrap median', 'CI lower', 'CI upper'])


## 3. Validation sensitivity

Alpha, absolute trajectory-weight clipping and soft target probability are varied one at a time. Equal values across clipping settings can mean no weights were clipped; that is not proof of robustness.

In [ ]:
pd.read_csv(output / 'data/sensitivity_analysis.csv')


## 4. Diagnostic figures

Mortality plots show simulated associations by mean action-bin discrepancy, not physical dose differences or causal treatment effects. Counts and Wilson intervals expose sparse groups.

In [ ]:
for filename in ['policy_value_comparison.png', 'action_distribution.png', 'dose_excess_vs_mortality.png']:
    display(Image(filename=str(output / 'figures' / filename)))
display(pd.read_csv(output / 'figures/mortality_bin_summary.csv'))


## Presentation limits

This simplified 75-state/30-feature synthetic demonstration is inspired by the paper, not a clinical reproduction. Real-data extraction, confounding adjustment, prospective validation and HCOPE guarantees are not implemented. Reward is not survival percentage. Low ESS prevents a credible claim that the target policy outperforms clinicians.